In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

SILVER_DIR = "/Volumes/workspace/default/my_volume/drone_pipeline/data/silver"
GOLD_DIR = "/Volumes/workspace/default/my_volume/drone_pipeline/data/gold"

In [0]:
def get_spark():
    return SparkSession.builder.appName("DroneGoldLayer").getOrCreate()

In [0]:
def main():
    spark = get_spark()
    # setLogLevel needs SparkContext, which isn't available on Databricks
    # Serverless compute -- so we guard it and skip quietly if unsupported.
    try:
        spark.sparkContext.setLogLevel("ERROR")
    except Exception:
        pass

    drones = spark.read.parquet(f"{SILVER_DIR}/drones")
    deliveries = spark.read.parquet(f"{SILVER_DIR}/deliveries")
    logs = spark.read.parquet(f"{SILVER_DIR}/flight_logs")

    # ---- 1 & 2. Success rate + failure rate, per drone ----
    outcome_counts = (
        logs.groupBy("drone_id")
        .agg(
            F.count("*").alias("total_flights"),
            F.sum(F.when(F.col("status") == "SUCCESS", 1).otherwise(0)).alias("success_count"),
            F.sum("failure_flag").alias("failure_count"),
        )
        .withColumn("avg_success_rate", F.round(F.col("success_count") / F.col("total_flights") * 100, 2))
        .withColumn("failure_rate", F.round(F.col("failure_count") / F.col("total_flights") * 100, 2))
    )

    drone_kpis = drones.join(outcome_counts, on="drone_id", how="left").select(
        "drone_id", "model", "max_range_km", "total_flights",
        "avg_success_rate", "failure_rate",
    )
    drone_kpis.write.format("parquet").mode("overwrite").save(f"{GOLD_DIR}/drone_kpis")
    print(f"[GOLD] drone_kpis -> {drone_kpis.count()} rows")

    # ---- 3. Average delivery time, per drone ----
    avg_delivery_time = (
        deliveries.groupBy("drone_id")
        .agg(F.round(F.avg("delivery_duration"), 2).alias("avg_delivery_time_minutes"))
    )
    avg_delivery_time.write.format("parquet").mode("overwrite").save(f"{GOLD_DIR}/avg_delivery_time")
    print(f"[GOLD] avg_delivery_time -> {avg_delivery_time.count()} rows")

    # ---- 4. Battery efficiency: distance_km / battery_consumed, averaged per drone ----
    battery_efficiency = (
        deliveries.withColumn(
            "efficiency_km_per_pct",
            F.round(F.col("distance_km") / F.col("battery_consumed"), 3),
        )
        .groupBy("drone_id")
        .agg(F.round(F.avg("efficiency_km_per_pct"), 3).alias("avg_battery_efficiency"))
    )
    battery_efficiency.write.format("parquet").mode("overwrite").save(f"{GOLD_DIR}/battery_efficiency")
    print(f"[GOLD] battery_efficiency -> {battery_efficiency.count()} rows")

    # ---- 5. High-risk zones: failure count per destination ----
    high_risk_zones = (
        logs.filter(F.col("failure_flag") == 1)
        .groupBy("destination")
        .agg(F.count("*").alias("failure_count"))
        .orderBy(F.desc("failure_count"))
    )
    high_risk_zones.write.format("parquet").mode("overwrite").save(f"{GOLD_DIR}/high_risk_zones")
    print(f"[GOLD] high_risk_zones -> {high_risk_zones.count()} rows")

    print("\nGold layer complete. KPI tables are ready for BI / SQL reporting.")

    print("\n--- Preview: drone_kpis ---")
    drone_kpis.orderBy(F.desc("failure_rate")).show(5, truncate=False)

    print("--- Preview: high_risk_zones ---")
    high_risk_zones.show(5, truncate=False)

    spark.stop()


if __name__ == "__main__":
    main()


[GOLD] drone_kpis -> 25 rows
[GOLD] avg_delivery_time -> 25 rows
[GOLD] battery_efficiency -> 25 rows
[GOLD] high_risk_zones -> 10 rows

Gold layer complete. KPI tables are ready for BI / SQL reporting.

--- Preview: drone_kpis ---
+--------+-----------+------------+-------------+----------------+------------+
|drone_id|model      |max_range_km|total_flights|avg_success_rate|failure_rate|
+--------+-----------+------------+-------------+----------------+------------+
|2       |CargoWing  |18.9        |29           |68.97           |31.03       |
|7       |CargoWing  |33.9        |15           |73.33           |26.67       |
|15      |AeroMule   |42.2        |21           |76.19           |23.81       |
|12      |SkyHawk200 |58.1        |22           |77.27           |22.73       |
|14      |SwiftDrone3|19.4        |29           |79.31           |20.69       |
+--------+-----------+------------+-------------+----------------+------------+
only showing top 5 rows
--- Preview: high_risk_z